# Privacy Trade-off Experiments

Run the same baseline training setup across different privacy filters, then compare test accuracy, macro F1, and loss.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from PIL import Image

if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().resolve().parent
else:
    PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.configs import (
    DEFAULT_DATA_ROOT,
    RESULTS_MODELS_DIR,
    RESULTS_PLOTS_DIR,
    SRC_DIR,
    BaselineExperimentConfig,
    build_train_command,
    format_intensity,
    resolve_python_bin,
)

PYTHON_BIN = resolve_python_bin()
TRAIN_SCRIPT = SRC_DIR / "train_baseline.py"
DATA_ROOT = DEFAULT_DATA_ROOT
TRADEOFF_PLOT_PATH = RESULTS_PLOTS_DIR / "privacy_tradeoff_summary.png"

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

print("Project root:", PROJECT_ROOT)
print("Python used for training:", PYTHON_BIN)
print("Dataset root:", DATA_ROOT)

## Experiment Setup

Keep `RUN_EXPERIMENTS = False` while reviewing the grid. Set it to `True` when you are ready to launch the runs.

In [ ]:
RUN_EXPERIMENTS = False
SKIP_COMPLETED = True

MODEL_NAME = "resnet18"
EPOCHS = 5
BATCH_SIZE = 64
IMAGE_SIZE = 224
NUM_WORKERS = 4
PIN_MEMORY = True
PERSISTENT_WORKERS = True
PREFETCH_FACTOR = 2
MAX_SAMPLES_PER_SPLIT = None

PRIVACY_EXPERIMENTS = [
    {"privacy_mode": "none", "privacy_intensity": 0},
    {"privacy_mode": "blur", "privacy_intensity": 3},
    {"privacy_mode": "blur", "privacy_intensity": 5},
    {"privacy_mode": "blur", "privacy_intensity": 9},
    {"privacy_mode": "edges", "privacy_intensity": 0},
    {"privacy_mode": "noise", "privacy_intensity": 50},
    {"privacy_mode": "noise", "privacy_intensity": 100},
    {"privacy_mode": "noise", "privacy_intensity": 250},
]


def make_run_suffix(privacy_mode, privacy_intensity):
    intensity_tag = format_intensity(privacy_intensity)
    return f"tradeoff_{privacy_mode}_{intensity_tag}_nw{NUM_WORKERS}"


def make_config(experiment):
    return BaselineExperimentConfig.gpu_tuned_config(
        model=MODEL_NAME,
        weights="pretrained",
        privacy_mode=experiment["privacy_mode"],
        privacy_intensity=experiment["privacy_intensity"],
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        image_size=IMAGE_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
        prefetch_factor=PREFETCH_FACTOR,
        max_samples_per_split=MAX_SAMPLES_PER_SPLIT,
        run_suffix=make_run_suffix(
            experiment["privacy_mode"],
            experiment["privacy_intensity"],
        ),
    )


configs = [make_config(experiment) for experiment in PRIVACY_EXPERIMENTS]

experiment_rows = []
for config in configs:
    experiment_rows.append(
        {
            "run_name": config.run_name,
            "privacy_mode": config.privacy_mode,
            "privacy_intensity": config.privacy_intensity,
            "epochs": config.epochs,
            "batch_size": config.batch_size,
            "image_size": config.image_size,
            "num_workers": config.num_workers,
            "metrics_exists": config.metrics_path.exists(),
        }
    )

experiments_df = pd.DataFrame(experiment_rows)
experiments_df

In [ ]:
required_paths = {
    "train_script": TRAIN_SCRIPT,
    "data_root": DATA_ROOT,
    "venv_python": PYTHON_BIN,
    "results_models_dir": RESULTS_MODELS_DIR,
    "results_plots_dir": RESULTS_PLOTS_DIR,
}

for name, path in required_paths.items():
    print(f"{name}: {path} | exists={path.exists()}")

train_folders = sorted(path.name for path in (DATA_ROOT / "train").iterdir() if path.is_dir())
print("Train classes:", train_folders)

In [ ]:
command_rows = []
for config in configs:
    command = build_train_command(
        config=config,
        train_script=TRAIN_SCRIPT,
        python_bin=PYTHON_BIN,
        data_root=DATA_ROOT,
    )
    command_rows.append(
        {
            "run_name": config.run_name,
            "privacy_mode": config.privacy_mode,
            "privacy_intensity": config.privacy_intensity,
            "metrics_path": str(config.metrics_path),
            "command": subprocess.list2cmdline(command),
        }
    )

commands_df = pd.DataFrame(command_rows)
commands_df[["run_name", "privacy_mode", "privacy_intensity", "metrics_path"]]

## Run Experiments

This cell streams training logs in real time. Completed runs are skipped when `SKIP_COMPLETED = True`.

In [ ]:
def stream_training(command):
    process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )

    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")

    return process.wait()


if not RUN_EXPERIMENTS:
    print("RUN_EXPERIMENTS is False. Set it to True in the setup cell when ready.")
else:
    suite_start = time.time()
    for index, config in enumerate(configs, start=1):
        if SKIP_COMPLETED and config.metrics_path.exists():
            print(f"Skipping completed run {index}/{len(configs)}: {config.run_name}")
            continue

        command = build_train_command(
            config=config,
            train_script=TRAIN_SCRIPT,
            python_bin=PYTHON_BIN,
            data_root=DATA_ROOT,
        )

        print(f"\nStarting run {index}/{len(configs)}: {config.run_name}")
        print(subprocess.list2cmdline(command))
        return_code = stream_training(command)
        if return_code != 0:
            raise RuntimeError(f"Training failed for {config.run_name} with return code {return_code}.")

    elapsed_minutes = (time.time() - suite_start) / 60
    print(f"\nExperiment suite finished in {elapsed_minutes:.2f} minutes.")

## Compare Results

In [ ]:
def load_metrics(config):
    if not config.metrics_path.exists():
        return None

    payload = json.loads(config.metrics_path.read_text(encoding="utf-8"))
    test_metrics = payload.get("test_metrics", {})
    history = payload.get("history", {})
    val_history = history.get("val", [])
    best_epoch = payload.get("best_epoch")

    row = {
        "run_name": payload.get("run_name", config.run_name),
        "model": payload.get("model", config.model),
        "privacy_mode": payload.get("privacy_mode", config.privacy_mode),
        "privacy_intensity": payload.get("privacy_intensity", config.privacy_intensity),
        "run_suffix": payload.get("run_suffix", config.run_suffix),
        "epochs": config.epochs,
        "best_epoch": best_epoch,
        "batch_size": payload.get("batch_size", config.batch_size),
        "image_size": payload.get("image_size", config.image_size),
        "num_workers": payload.get("num_workers", config.num_workers),
        "pin_memory": payload.get("pin_memory", config.pin_memory),
        "persistent_workers": payload.get("persistent_workers", config.persistent_workers),
        "prefetch_factor": payload.get("prefetch_factor", config.prefetch_factor),
        "metrics_path": str(config.metrics_path),
    }
    row.update(test_metrics)

    if best_epoch is not None and val_history:
        best_index = max(min(best_epoch - 1, len(val_history) - 1), 0)
        row["best_val_f1_macro"] = val_history[best_index].get("f1_macro")
        row["best_val_accuracy"] = val_history[best_index].get("accuracy")

    return row


rows = [load_metrics(config) for config in configs]
completed_rows = [row for row in rows if row is not None]
missing_runs = [config.run_name for config, row in zip(configs, rows) if row is None]

summary_df = pd.DataFrame(completed_rows)
if not summary_df.empty:
    summary_df = summary_df.sort_values(["privacy_mode", "privacy_intensity"]).reset_index(drop=True)

if missing_runs:
    print("Missing metrics for:")
    for run_name in missing_runs:
        print("-", run_name)

summary_df

In [ ]:
if summary_df.empty:
    print("No completed experiment metrics found yet.")
else:
    baseline_rows = summary_df[
        (summary_df["privacy_mode"] == "none")
        & (summary_df["privacy_intensity"].astype(float) == 0.0)
    ]

    if baseline_rows.empty:
        print("No baseline row found. Deltas will not be computed yet.")
        comparison_df = summary_df.copy()
        comparison_df["accuracy_delta_vs_baseline"] = None
        comparison_df["f1_macro_delta_vs_baseline"] = None
        comparison_df["loss_delta_vs_baseline"] = None
    else:
        baseline = baseline_rows.iloc[0]
        comparison_df = summary_df.copy()
        comparison_df["accuracy_delta_vs_baseline"] = comparison_df["accuracy"] - baseline["accuracy"]
        comparison_df["f1_macro_delta_vs_baseline"] = comparison_df["f1_macro"] - baseline["f1_macro"]
        comparison_df["loss_delta_vs_baseline"] = comparison_df["loss"] - baseline["loss"]

    display(
        comparison_df[
            [
                "run_name",
                "privacy_mode",
                "privacy_intensity",
                "accuracy",
                "f1_macro",
                "loss",
                "accuracy_delta_vs_baseline",
                "f1_macro_delta_vs_baseline",
                "loss_delta_vs_baseline",
                "best_epoch",
            ]
        ]
    )

In [ ]:
if summary_df.empty:
    print("No metrics available to plot yet.")
else:
    plot_df = summary_df.copy()
    plot_df["label"] = plot_df.apply(
        lambda row: f"{row['privacy_mode']}\n{row['privacy_intensity']}",
        axis=1,
    )

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    colors = {
        "none": "#2f4858",
        "blur": "#4f772d",
        "edges": "#9a031e",
        "noise": "#f77f00",
    }
    bar_colors = [colors.get(mode, "#355070") for mode in plot_df["privacy_mode"]]

    axes[0].bar(plot_df["label"], plot_df["accuracy"], color=bar_colors)
    axes[0].set_title("Test Accuracy")
    axes[0].set_ylim(max(0.0, plot_df["accuracy"].min() - 0.05), 1.0)

    axes[1].bar(plot_df["label"], plot_df["f1_macro"], color=bar_colors)
    axes[1].set_title("Test Macro F1")
    axes[1].set_ylim(max(0.0, plot_df["f1_macro"].min() - 0.05), 1.0)

    axes[2].bar(plot_df["label"], plot_df["loss"], color=bar_colors)
    axes[2].set_title("Test Loss")
    axes[2].set_ylim(0.0, plot_df["loss"].max() * 1.15)

    for axis in axes:
        axis.tick_params(axis="x", rotation=45)
        axis.grid(axis="y", alpha=0.25)

    fig.suptitle(f"Privacy vs Performance - {MODEL_NAME}", fontsize=16)
    plt.tight_layout()
    TRADEOFF_PLOT_PATH.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(TRADEOFF_PLOT_PATH, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved plot to:", TRADEOFF_PLOT_PATH)

In [ ]:
if TRADEOFF_PLOT_PATH.exists():
    display(Image.open(TRADEOFF_PLOT_PATH))
else:
    print("Trade-off plot not found yet.")